**Dim Customer**

In [0]:
%sql
select * from datamodeling.silver.silver_tabel; 

In [0]:
# %sql
# select *,row_number() over (order by customer_id) DimCustomerKey
# from (
# select distinct(customer_id) customer_id,  customer_email,customer_name,upper_name from datamodeling.silver.silver_tabel )

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS datamodeling.gold

In [0]:
%sql
create or replace table datamodeling.gold.DimCustomers
as
with remove_dup as 
(select distinct(customer_id) customer_id,  customer_email,customer_name,upper_name from datamodeling.silver.silver_tabel )
select *,row_number() over (order by customer_id) DimCustomerKey from remove_dup;

In [0]:
%sql
select * from datamodeling.gold.dimcustomers

**Dim Products**

In [0]:
%sql
create or replace table datamodeling.gold.DimProducts
as
with remove_dup as 
(select
distinct(product_id) product_id,
product_name,
product_category
from datamodeling.silver.silver_tabel)
select *,row_number() over (order by product_id) DimProductKey from remove_dup;

In [0]:
%sql
select * from datamodeling.gold.DimProducts

**Dim Payments**

In [0]:
%sql
create or replace table datamodeling.gold.DimPayment
as
with remove_dup as 
(select 
distinct(payment_type) payment_type
from datamodeling.silver.silver_tabel)
select *,row_number() over (order by payment_type) DimPaymentKey from remove_dup;

In [0]:
%sql
select * from datamodeling.gold.dimpayment

**Dim Region**

In [0]:
%sql
create or replace table datamodeling.gold.Dimcountry
as
with remove_dup as 
(select 
distinct(country) country
from datamodeling.silver.silver_tabel)
select *,row_number() over (order by country) DimCountryKey from remove_dup;

In [0]:
%sql
select * from datamodeling.gold.DimCountry

**Dim Sales**

In [0]:
%sql
create or replace table datamodeling.gold.Dimsales
as
select
row_number() over (order by order_id) DimSalesKey,
order_id,
 order_date,
 customer_id,
 customer_name,
 customer_email,
 product_id,
 product_name,
 product_category,
 payment_type,
 country,
 last_updated,
 upper_name,
 processed_date
 from datamodeling.silver.silver_tabel

In [0]:
%sql
select * from datamodeling.gold.Dimsales;

**Fact Table**

In [0]:
%sql
create or replace table datamodeling.gold.factSales
as
select
c.DimCustomerKey,
pr.DimProductKey,
p.DimPaymentKey,
co.DimCountryKey,
s.DimSalesKey,
f.quantity,
f.unit_price
 from datamodeling.silver.silver_tabel f
 left join 
 datamodeling.gold.dimcustomers c
 on f.customer_id = c.customer_id
 left join
 datamodeling.gold.dimproducts pr
 on f.product_id = pr.product_id
 left join
 datamodeling.gold.dimpayment p
 on f.payment_type = p.payment_type
 left join
 datamodeling.gold.dimcountry co
 on f.country = co.country
 left join
 datamodeling.gold.dimsales s
 on f.order_id = s.order_id


In [0]:
%sql
select * from datamodeling.gold.factsales

In [0]:
# %sql
# drop table datamodeling.gold.DimCustomers;
# drop table datamodeling.gold.DimProducts;
# drop table datamodeling.gold.Dimcountry;
# drop table datamodeling.gold.DimPayment;
# drop table datamodeling.gold.Dimsales;
# drop table datamodeling.gold.factSales;